# Suspicious username collection & first mention — r/TheseFuckingAccounts

Builds the seed dataset for Reddit Project 2: every username that a poster in
r/TheseFuckingAccounts **accused** of being a bot / spammer / karma farmer, when that
account was first called out, how many separate posts called it out, and whether the
account is currently **active** or **suspended**.

**Input** `/data/Emily_data/TheseFuckingAccounts_submissions.csv` — 17,631 submissions, 2014-12-21 → 2022-12-31.

**Output** `data/suspicious_usernames.parquet` — one row per suspicious account:

| column | meaning |
|---|---|
| `username` | account name as most commonly written in the posts |
| `username_key` | lowercased name — the join key (Reddit names are case-insensitive) |
| `first_mention_utc` | raw Unix timestamp of the earliest post naming this account |
| `first_mention_date` | `first_mention_utc` as a UTC datetime, for reading |
| `times_mentioned` | number of **distinct** posts naming this account (1 if only ever once) |
| `first_post_id` | submission id of that earliest post — lets you trace any row back to source |
| `extraction_method` | which rule matched, so low-confidence rows can be audited or dropped |
| `account_status` | `active` or `suspended` (everything else is dropped) |

**The accuser/victim distinction.** These posts routinely name two accounts: the suspected
bot *and* the original poster it stole from. Only the accused is collected. In
`u/testuser copied this post from u/notme`, only `testuser` is kept — see the
victim-exclusion rules and their tests in Step 2 and Step 3.

**Run order.** Steps 1-4 are cheap (seconds). Step 5 calls the Reddit API once per account
and is the slow part — it is checkpointed and resumable, and defaults to a sample.
Read the note in Step 5 before running it.

## Step 0 — Imports and configuration

In [ ]:
# Standard scientific stack used across this project.
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Reddit API client, and its exception module (needed to tell "deleted" from "suspended").
import praw
import prawcore

# Reads the credentials file so keys never appear in this notebook.
from dotenv import load_dotenv

# Show full frames while inspecting - project convention.
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# Usernames get truncated by the default 50-char column width, which hides extraction bugs.
pd.set_option('display.max_colwidth', 120)

print('pandas', pd.__version__, '| praw', praw.__version__)

In [ ]:
# ---- Paths -------------------------------------------------------------------------
# Source corpus: every r/TheseFuckingAccounts submission, one row per post.
SUBMISSIONS_CSV = Path('/data/Emily_data/TheseFuckingAccounts_submissions.csv')

# Credentials for the Reddit API. Lives outside the repo on purpose; never commit it.
ENV_PATH = Path('/home/eeporter/Reddit_project1_notebooks/reddit_env_vars.env')

# Output folder, created next to this notebook. Git-ignored (see .gitignore) because it
# holds collected account data.
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

# Final dataset.
OUTPUT_PARQUET = DATA_DIR / 'suspicious_usernames.parquet'
# Partial API results, written as we go so an interrupted run resumes instead of restarting.
STATUS_CHECKPOINT = DATA_DIR / 'account_status_checkpoint.parquet'

print('writing to', DATA_DIR.resolve())

## Step 1 — Load the submissions

In [ ]:
# low_memory=False: the file has 124 columns of mixed dtypes; chunked type inference
# otherwise emits DtypeWarning and can type the same column differently per chunk.
submissions = pd.read_csv(SUBMISSIONS_CSV, low_memory=False)

# Only four columns matter here. Keeping a narrow frame makes every later step faster.
#   id          - unique submission id, used to count *distinct* accusing posts
#   title       - many accusations name the account in the title only
#   selftext    - the post body, where most u/ mentions live
#   created_utc - post time, used for the first-mention date
posts = submissions[['id', 'title', 'selftext', 'created_utc']].copy()

# selftext is NaN for link posts (3,162 of them) and title is never null, but fill both:
# the regex helpers below expect strings, and NaN would raise.
posts['title'] = posts['title'].fillna('')
posts['selftext'] = posts['selftext'].fillna('')

# created_utc is read as float when NaNs are present elsewhere in the file; force integer
# seconds so min() returns a clean timestamp.
posts['created_utc'] = posts['created_utc'].astype('int64')

print(f'{len(posts):,} submissions')
print('date range:',
      pd.to_datetime(posts["created_utc"].min(), unit="s"), '->',
      pd.to_datetime(posts["created_utc"].max(), unit="s"))
posts.head(3)

In [ ]:
# ---- Collapse duplicate posts ------------------------------------------------------
# times_mentioned must count *separate* posts, so an identical submission posted twice
# must not count twice. All 17,631 ids are unique, so dedupe on content instead:
# identical (title, selftext) pairs are the same accusation appearing more than once.
# Keep the earliest copy, which is also the one that sets the first-mention date.
before = len(posts)
posts = (posts.sort_values('created_utc')
              .drop_duplicates(subset=['title', 'selftext'], keep='first')
              .reset_index(drop=True))
print(f'{before:,} posts -> {len(posts):,} after dropping {before - len(posts):,} duplicate submissions')
# Note: nearly all of these are placeholder pairs such as ("[deleted by user]", "[removed]"),
# which contain no usernames and so would not have affected counts anyway.

## Step 2 — Username extraction rules

Four things make this harder than a single `u/(\w+)` regex, all of them observed in this corpus:

1. **Two accounts per post.** The template `[Here](...) it copied/pasted /u/example_victim's
   submission/title from [here](...)` names the *victim*, not the bot. Possessive
   (`u/name's`) and `from u/name` / `by u/name` forms are therefore treated as victims and excluded.
2. **Markdown escaping.** Reddit escapes underscores, so `u/Example\_Suspicious` must be
   unescaped first or the name truncates to `One`.
3. **`u/` appears inside unrelated URLs.** `imgur.com/gallery/AbCdE/comment/123456`
   contains the substring `u/comment`. A left boundary is required so `u/` only matches
   at a word start.
4. **Many accusations never write `u/` at all** — `ExampleTeam - Born January 4, 2018, woke
   up June 5th`. These bare-name forms are matched only in tightly anchored positions,
   because an unanchored bare-word rule pulls in ordinary English ("also born on...").

Every row records which rule matched in `extraction_method`, so the low-confidence
bare-name rules can be audited or filtered out later.

In [ ]:
# Reddit usernames: 3-20 characters, letters/digits/underscore/hyphen.
NAME = r'[A-Za-z0-9_-]{3,20}'

# Reddit escapes markdown punctuation in stored text: "Example\_Suspicious".
# Undo that before matching or names silently truncate at the backslash.
MD_ESCAPE = re.compile(r'\\([_*~`\[\]()])')

def unescape_markdown(text):
    """Turn Reddit's escaped markdown ("a\\_b") back into plain text ("a_b")."""
    return MD_ESCAPE.sub(r'\1', str(text))


# A username mention, in either of the two forms that appear in this corpus:
#   1. a full profile URL:  https://www.reddit.com/user/NAME  (also old./np./amp. subdomains, /u/ form)
#   2. a bare mention:      u/NAME  or  /u/NAME
# The (?<![A-Za-z0-9_./]) lookbehind on form 2 is what stops "AbCdE/comment" matching as
# "u/comment": it requires the "u" to start a word rather than end one.
MENTION_RE = re.compile(
    r'(?:(?:https?://)?(?:[a-z0-9-]+\.)*reddit\.com/u(?:ser)?/(' + NAME + r'))'
    r'|(?:(?<![A-Za-z0-9_./])/?u/(' + NAME + r'))',
    re.IGNORECASE)

# ---- Victim patterns: accounts named as the *target* of the theft, not the accused ----
# Possessive: "copied/pasted /u/example_victim's submission/title" -> example_victim is the victim.
VICTIM_POSSESSIVE_RE = re.compile(
    r'(?<![A-Za-z0-9_./])/?u(?:ser)?/(' + NAME + r")\s*(?:'|\u2019)s\b",
    re.IGNORECASE)

# Attribution: "stolen from u/x", "originally by u/x", "credit to u/x" -> x is the victim.
# A bare "by" is deliberately NOT a victim marker: in this subreddit "spam posted by u/x"
# usually names the *accused*, and treating it as a victim marker also wrongly excluded the
# thief in "stolen from u/original by u/thief". Only "original(ly) by" marks a victim.
VICTIM_ATTRIBUTION_RE = re.compile(
    r'\b(?:from|stolen\s+from|belongs\s+to|credits?\s+to|original(?:ly)?\s+(?:by|from))\s+'
    r'(?:the\s+)?(?:real\s+)?(?:(?:https?://)?(?:[a-z0-9-]+\.)*reddit\.com)?/?u(?:ser)?/(' + NAME + r')\b',
    re.IGNORECASE)

# Names that are never a suspect account.
RESERVED_NAMES = {
    'deleted', 'removed', 'automoderator', 'reddit', 'me', 'all',
    'user', 'users', 'profile', 'account',
}

In [ ]:
def find_mentions(text):
    """Every username mentioned in `text`, in order, regardless of accused/victim role."""
    # MENTION_RE has two alternatives, so exactly one of the two groups is filled per match.
    return [m.group(1) or m.group(2) for m in MENTION_RE.finditer(text)]


def find_victims(text):
    """Lowercased names that `text` frames as the *original* poster, i.e. not suspects."""
    victims = set()
    for pattern in (VICTIM_POSSESSIVE_RE, VICTIM_ATTRIBUTION_RE):
        for m in pattern.finditer(text):
            victims.add(m.group(1).lower())
    return victims

In [ ]:
# ---- Bare-name rules (no "u/" prefix) ----------------------------------------------
# Each is anchored to the start of the title, where the accused account is named in this
# subreddit's house style. Anchoring is what keeps ordinary words out: in
# "/u/example_acct35 - also born on January 1", an unanchored "word before 'born'" rule
# captures "also", while an anchored one correctly captures nothing (the real name is
# already picked up by MENTION_RE).

# Reddit's own profile page title: "overview for SomeAccount".
TITLE_OVERVIEW_RE = re.compile(r'^\s*overview for (' + NAME + r')\b', re.IGNORECASE)

# "ExampleTheName - born on January 3, 2017 and just woke up after a long break."
# This is the "born on x, woke up on xx" phrasing: a dormant account that suddenly activates.
TITLE_BORN_RE = re.compile(
    r'^\s*(' + NAME + r')\s*[:\u2014\u2013-]+\s*(?:was\s+)?born\b',
    re.IGNORECASE)

# "example-101 - 7 days old, submissions/titles copied verbatim"
TITLE_AGE_RE = re.compile(
    r'^\s*(' + NAME + r')\s*[:\u2014\u2013-]+\s*\d+\s+(?:second|minute|hour|day|week|month|year)s?\s+old\b',
    re.IGNORECASE)

# "exampliaz: \"word salad\" post titles in r/aww" - a bare name, then a colon/dash, then prose.
# This is the loosest rule and would match plenty of ordinary sentences, so a match is only
# accepted when the same name is also written as an explicit u/ mention somewhere in the
# corpus (the `vocabulary` argument). That cross-check is built in the next cell.
TITLE_LEAD_RE = re.compile(r'^\s*(' + NAME + r')\s*[:\u2014\u2013-]+\s')

# Ordinary words that lead titles and would otherwise slip through the bare-name rules.
STOPWORD_LEADS = {
    'also', 'both', 'one', 'another', 'this', 'that', 'these', 'those', 'the', 'and',
    'but', 'now', 'new', 'here', 'there', 'they', 'them', 'his', 'her', 'its', 'it',
    'who', 'what', 'why', 'how', 'when', 'again', 'still', 'more', 'psa', 'update',
    'edit', 'help', 'look', 'found', 'guy', 'bot', 'bots', 'spam', 'spammer', 'http',
    'https', 'www', 'com', 'net', 'org',
}

def _acceptable_bare_name(name):
    """Reject bare-name matches that are obviously ordinary words or reserved."""
    key = name.lower()
    return key not in STOPWORD_LEADS and key not in RESERVED_NAMES

In [ ]:
def build_vocabulary(frame):
    """
    Lowercased names that appear *somewhere* in the corpus as an explicit u/ mention.

    Used to gate TITLE_LEAD_RE: a bare leading word is only accepted as a username if some
    other post wrote it with a u/ prefix, which is strong evidence it is a real account.
    """
    vocabulary = set()
    for title, selftext in zip(frame['title'], frame['selftext']):
        for name in find_mentions(unescape_markdown(title)) + find_mentions(unescape_markdown(selftext)):
            key = name.lower()
            if key not in RESERVED_NAMES:
                vocabulary.add(key)
    return vocabulary


VOCABULARY = build_vocabulary(posts)
print(f'{len(VOCABULARY):,} distinct names written with an explicit u/ prefix somewhere in the corpus')

In [ ]:
def extract_suspects(title, selftext, vocabulary=frozenset()):
    """
    Return {lowercased_name: (extraction_method, name_as_written)} for one submission.

    Only accounts the poster is *accusing* are returned; accounts named as the victim of
    a repost are excluded. A name is returned at most once per post no matter how many
    times it is mentioned, so `times_mentioned` counts posts and not raw mentions.
    """
    title = unescape_markdown(title)
    selftext = unescape_markdown(selftext)

    # Victim names are computed across the whole post: a name introduced as the original
    # author in the body should not be collected just because it also appears in the title.
    victims = find_victims(title) | find_victims(selftext)

    found = {}

    # Rule 1 - explicit u/ or profile-URL mentions. Highest confidence, and the source of
    # the large majority of rows. Title first so the name-as-written prefers the title form.
    for name in find_mentions(title) + find_mentions(selftext):
        key = name.lower()
        if key in RESERVED_NAMES or key in victims:
            continue
        found.setdefault(key, ('explicit_mention', name))

    # Rules 2-4 - anchored bare-name patterns in the title.
    for method, pattern in (('title_overview', TITLE_OVERVIEW_RE),
                            ('title_born',     TITLE_BORN_RE),
                            ('title_age',      TITLE_AGE_RE)):
        m = pattern.search(title)
        if m and _acceptable_bare_name(m.group(1)):
            key = m.group(1).lower()
            if key not in victims:
                found.setdefault(key, (method, m.group(1)))

    # Rule 5 - loose leading bare name, accepted only if corroborated by the vocabulary.
    m = TITLE_LEAD_RE.search(title)
    if m and _acceptable_bare_name(m.group(1)) and m.group(1).lower() in vocabulary:
        key = m.group(1).lower()
        if key not in victims:
            found.setdefault(key, ('title_lead_corroborated', m.group(1)))

    return found

## Step 3 — Tests for the extraction rules

Each assertion below encodes one requirement. They run in a second and fail loudly, so
re-run this cell after any change to the regexes above.

In [ ]:
def suspects_of(title, selftext='', vocabulary=frozenset()):
    """Test helper: just the lowercased suspect names for a title/body pair."""
    return set(extract_suspects(title, selftext, vocabulary))


# --- Requirement: collect the accused, not the account they stole from ----------------
# This is the exact example from the project brief.
assert suspects_of('', 'u/testusername copied this post from u/nottestname') == {'testusername'}, \
    'the "copied ... from u/x" victim must be excluded'

# The dominant template in this corpus: possessive marks the victim.
assert suspects_of('', 'https://www.reddit.com/user/example_accused_b\n\n[Here](x) it copied/pasted '
                       "/u/example_victim's submission/title from [here](y).") == {'example_accused_b'}, \
    "possessive /u/name's must be read as the victim"

assert suspects_of('', 'stolen from u/original_artist by u/thief_account') == {'thief_account'}, \
    '"stolen from u/x" must exclude x'

# --- Requirement: "born on x, woke up on xx" phrasing -------------------------------
assert suspects_of('ExampleTheName - born on January 3, 2017 and just woke up after a long break.') \
    == {'examplethename'}, 'born/woke phrasing must yield the bare leading name'
assert suspects_of('ExampleTeam - Born January 4, 2018, woke up June 5th. Copy/paste bot.') \
    == {'exampleteam'}, 'capitalised "Born" must match too'

# ...but the same phrasing must NOT capture the ordinary word before "born" when the real
# account is already named with a u/ prefix.
assert suspects_of('/u/example_acct35 - also born on January 1, 2015, woke up 9 days ago.') \
    == {'example_acct35'}, '"also born on" must not yield "also"'
assert suspects_of('/u/ExampleOne and /u/ExampleTwo, both born on January 2, both woke up 4-5 hours ago.') \
    == {'exampleone', 'exampletwo'}, '"both born on" must not yield "both"'

# --- Requirement: markdown escapes must not truncate names --------------------------
assert suspects_of('', '[https://old.reddit.com/user/Example\\_Suspicious](https://old.reddit.com/user/Example\\_Suspicious)') \
    == {'example_suspicious'}, 'escaped underscore must be unescaped before matching'

# --- Requirement: "u/" inside an unrelated URL is not a mention ---------------------
assert suspects_of('', 'its comment on [this imgur page](https://imgur.com/gallery/AbCdE/comment/1234567890)') \
    == set(), 'a trailing "u" in a path segment must not create a u/ mention'
assert suspects_of('', 'see https://www.reddit.com/r/pics/comments/7jy60m/some_title/') == set(), \
    'an ordinary submission link contains no username'

# --- Requirement: reddit profile URL variants all resolve ---------------------------
for url in ['https://www.reddit.com/user/SpamBot99', 'http://old.reddit.com/u/SpamBot99',
            'reddit.com/user/SpamBot99', 'u/SpamBot99', '/u/SpamBot99']:
    assert suspects_of('', url) == {'spambot99'}, f'failed to parse {url}'

# --- Requirement: multi-account "sewage report" posts collect every listed account ---
assert suspects_of('Crypto spammer report', 'u/Example_Word1234\n\nu/examplekeysmash\n\nu/ExampleWord31') \
    == {'example_word1234', 'examplekeysmash', 'exampleword31'}, 'list posts must yield all accounts'

# --- Requirement: reserved / placeholder names are never collected ------------------
assert suspects_of('', 'u/AutoModerator removed it, u/[deleted] posted it') == set(), \
    'AutoModerator and deleted must be excluded'

# --- Requirement: one row per post per account, however many times it is mentioned ---
assert extract_suspects('u/dupe_bot', 'u/dupe_bot again, and u/dupe_bot once more') \
    .keys() == {'dupe_bot'}, 'repeat mentions inside one post must collapse'

# --- Requirement: the vocabulary gate on the loosest rule ---------------------------
assert suspects_of('exampliaz: "word salad" post titles in r/aww') == set(), \
    'uncorroborated bare lead name must be rejected'
assert suspects_of('exampliaz: "word salad" post titles in r/aww', vocabulary={'exampliaz'}) == {'exampliaz'}, \
    'the same name is accepted once corroborated by an explicit mention elsewhere'
assert suspects_of('Found a little network of bots and subs!', vocabulary={'found'}) == set(), \
    'a stopword lead must stay rejected even if it appears in the vocabulary'

print('all extraction tests passed')

## Step 4 — Build the username dataset

In [ ]:
# Walk every (deduplicated) post once and emit one record per accused account per post.
mention_records = []
for post in posts.itertuples(index=False):
    for key, (method, name_as_written) in extract_suspects(post.title, post.selftext, VOCABULARY).items():
        mention_records.append({
            'username_key': key,
            'username': name_as_written,
            'extraction_method': method,
            'post_id': post.id,
            'created_utc': post.created_utc,
        })

mentions = pd.DataFrame(mention_records)
print(f'{len(mentions):,} (account, post) mentions covering {mentions["username_key"].nunique():,} distinct accounts')
print()
print(mentions['extraction_method'].value_counts().to_string())

In [ ]:
# ---- Collapse to one row per account ------------------------------------------------
# Sorting by time first means .first() picks the earliest post for the first-mention fields.
mentions_sorted = mentions.sort_values('created_utc')

suspicious = (
    mentions_sorted
    .groupby('username_key')
    .agg(
        # nunique over post_id is what makes this a count of separate posts: a name
        # mentioned three times inside one post still counts once.
        times_mentioned=('post_id', 'nunique'),
        first_mention_utc=('created_utc', 'min'),
        first_post_id=('post_id', 'first'),
        extraction_method=('extraction_method', 'first'),
    )
    .reset_index()
)

# Display name: the spelling used most often across posts. Reddit names are case-insensitive
# for lookup, but the canonical casing is nicer to read and to publish.
preferred_spelling = (
    mentions.groupby(['username_key', 'username']).size()
            .sort_values(ascending=False)
            .reset_index(name='n')
            .drop_duplicates('username_key')
            .set_index('username_key')['username']
)
suspicious['username'] = suspicious['username_key'].map(preferred_spelling)

# Human-readable date alongside the raw timestamp. Project convention is to keep the raw
# Unix value in stored files and convert only for display, so both columns are kept.
suspicious['first_mention_date'] = pd.to_datetime(suspicious['first_mention_utc'], unit='s')

suspicious = suspicious[['username', 'username_key', 'first_mention_utc', 'first_mention_date',
                         'times_mentioned', 'first_post_id', 'extraction_method']]

print(f'{len(suspicious):,} distinct suspicious accounts')
print()
print('accounts by number of separate posts mentioning them:')
print(suspicious['times_mentioned'].value_counts().sort_index().head(10).to_string())
suspicious.sort_values('times_mentioned', ascending=False).head(10)

In [ ]:
# ---- Tests on the assembled table ---------------------------------------------------
# One row per account, no duplicates.
assert suspicious['username_key'].is_unique, 'username_key must be unique'
# The brief: a name seen once is 1, a name in two different posts is 2.
assert suspicious['times_mentioned'].min() >= 1, 'every account must be mentioned at least once'
assert (suspicious['times_mentioned'] <= posts.shape[0]).all(), 'count cannot exceed the post count'
# No empty or malformed names slipped through.
assert suspicious['username'].str.len().between(3, 20).all(), 'names must be 3-20 chars'
assert suspicious['username_key'].str.match(r'^[a-z0-9_-]+$').all(), 'keys must be lowercase name chars'
# first_mention_utc must really be the earliest post for that account.
_spot = mentions[mentions['username_key'] == suspicious.loc[0, 'username_key']]
assert _spot['created_utc'].min() == suspicious.loc[0, 'first_mention_utc'], 'first mention must be the min'
# Every first_post_id is a real post that really names that account.
_sample = suspicious.sample(min(50, len(suspicious)), random_state=0)
for row in _sample.itertuples():
    src = posts.loc[posts['id'] == row.first_post_id].iloc[0]
    assert row.username_key in extract_suspects(src.title, src.selftext, VOCABULARY), \
        f'{row.username_key} not recoverable from its own first post {row.first_post_id}'
print('all dataset tests passed')

In [ ]:
# Cross-check the counts by hand for the most-mentioned account: recount its distinct
# posts straight from the corpus and compare with the aggregated value.
top = suspicious.sort_values('times_mentioned', ascending=False).iloc[0]
manual = {p.id for p in posts.itertuples(index=False)
          if top.username_key in extract_suspects(p.title, p.selftext, VOCABULARY)}
print(f'{top.username}: table says {top.times_mentioned}, recount from corpus says {len(manual)}')
assert len(manual) == top.times_mentioned, 'independent recount disagrees with the aggregation'
print('first mentioned:', top.first_mention_date, f'(post {top.first_post_id})')
print('all count tests passed')

## Step 5 — Account status from the Reddit API

Three outcomes per lookup:

| API response | meaning | kept? |
|---|---|---|
| object has an `id` | account exists and is **active** | yes |
| `is_suspended` is `True` | account is **suspended** | yes |
| `404 NotFound` | deleted or never existed | **dropped** |

Anything else (deleted, shadowbanned-to-404, transient error) is dropped, per the brief.

**This step is slow.** A cold burst runs at ~6 accounts/s, but Reddit throttles sustained
use and the rate settles near ~2/s, so resolving all ~56,000 extracted accounts takes
on the order of 7-8 hours of wall clock. So:

* Results are checkpointed to `data/account_status_checkpoint.parquet` after every chunk.
  Re-running the cell **resumes** — already-checked accounts are skipped.
* `SAMPLE_SIZE` below caps how many accounts are checked in one pass. It defaults to a
  small sample so the notebook runs end to end in about a minute.
* **Set `SAMPLE_SIZE = None` and re-run to collect everything.** Interrupt it whenever;
  progress is kept, and running it again picks up where it stopped.

In [ ]:
# Credentials are read from the .env file, never written in this notebook.
# Key names come from that file: praw_script, praw_key, login_name, password, user_agent.
# (Note: the login key is `login_name`, not `user_name`.)
load_dotenv(ENV_PATH)

reddit_api = praw.Reddit(
    client_id=os.getenv('praw_script'),
    client_secret=os.getenv('praw_key'),
    username=os.getenv('login_name'),
    password=os.getenv('password'),
    user_agent=os.getenv('user_agent'),
    # On a 429, wait up to 6 minutes for the window to reset instead of raising.
    ratelimit_seconds=360,
)
# Nothing here writes to Reddit; read_only avoids any accidental POST.
reddit_api.read_only = True

# Fetching a known account proves the credentials loaded and the client works.
print('authenticated, read_only =', reddit_api.read_only)
print('smoke test - u/spez id:', reddit_api.redditor('spez').id)

In [ ]:
def check_account_status(username):
    """
    Look up one account and classify it.

    Returns 'active', 'suspended', 'deleted' (404), or 'error:<Type>' for anything
    transient. Only 'active' and 'suspended' survive into the final dataset; the other
    values are kept here so an interrupted or throttled run is distinguishable from a
    genuine 404 on the next pass.
    """
    try:
        redditor = reddit_api.redditor(username)
        # Attribute access triggers the actual HTTP fetch, so this line is where the
        # 404 for a deleted account is raised.
        attributes = vars(redditor)

        # Suspended accounts come back as a stub: is_suspended=True and no id.
        if getattr(redditor, 'is_suspended', False):
            return 'suspended'
        # A real, live account always carries an id.
        if 'id' in attributes:
            return 'active'
        # Fetched but neither suspended nor id-bearing: treat as not usable.
        return 'other'
    except prawcore.exceptions.NotFound:
        # Deleted account, or a username that never existed.
        return 'deleted'
    except prawcore.exceptions.Forbidden:
        return 'forbidden'
    except Exception as exc:  # network blip, throttle, 5xx - retry on a later pass
        return f'error:{type(exc).__name__}'

In [ ]:
# ---- Test the classifier against accounts with known, stable states -----------------
# u/spez is Reddit's CEO and will not vanish; the random string cannot exist.
assert check_account_status('spez') == 'active', 'a live account must classify as active'
assert check_account_status('thisaccountdoesnotexist99xyz') == 'deleted', \
    'a nonexistent account must classify as deleted'

# At least one account known to be suspended, drawn from this corpus.
_known_suspended = check_account_status('exampleknight33')
print('exampleknight33 ->', _known_suspended)
assert _known_suspended == 'suspended', 'expected a suspended classification'

# Case-insensitivity: Reddit resolves names regardless of case, so the lowercased
# username_key is a safe lookup value.
assert check_account_status('SpEz') == 'active', 'lookup must be case-insensitive'
print('all status-classifier tests passed')

In [ ]:
# How many accounts to check in this pass. None = every remaining account (hours).
SAMPLE_SIZE = 500
# Write the checkpoint every this many lookups so little is lost to an interrupt.
CHUNK = 100

# Resume: load whatever a previous pass already resolved.
if STATUS_CHECKPOINT.exists():
    checked = pd.read_parquet(STATUS_CHECKPOINT)
    # Transient failures are worth retrying; settled verdicts are not.
    settled = checked[~checked['account_status'].str.startswith('error')]
    print(f'resuming: {len(settled):,} accounts already resolved')
else:
    settled = pd.DataFrame(columns=['username_key', 'account_status'])
    print('no checkpoint, starting fresh')

todo = suspicious.loc[~suspicious['username_key'].isin(settled['username_key']), 'username_key'].tolist()
if SAMPLE_SIZE is not None:
    todo = todo[:SAMPLE_SIZE]

print(f'{len(todo):,} accounts to check this pass '
      f'({len(suspicious) - len(settled):,} outstanding overall)')

In [ ]:
# ---- The lookup loop ----------------------------------------------------------------
results = list(settled.itertuples(index=False, name=None))
started = time.time()

for i, name in enumerate(todo, start=1):
    results.append((name, check_account_status(name)))

    # Periodic checkpoint + progress line.
    if i % CHUNK == 0 or i == len(todo):
        pd.DataFrame(results, columns=['username_key', 'account_status']).to_parquet(
            STATUS_CHECKPOINT, index=False, engine='pyarrow', compression='zstd')
        rate = i / max(time.time() - started, 1e-9)
        remaining = (len(todo) - i) / max(rate, 1e-9)
        print(f'  {i:,}/{len(todo):,} checked  |  {rate:.1f}/s  |  ~{remaining/60:.1f} min left in this pass')

status = pd.DataFrame(results, columns=['username_key', 'account_status']).drop_duplicates('username_key')
print(f'\n{len(status):,} accounts resolved in total')
print(status['account_status'].value_counts().to_string())

In [ ]:
# ---- Join the status on and drop everything that is not active or suspended ---------
labelled = suspicious.merge(status, on='username_key', how='inner')

before = len(labelled)
final = labelled[labelled['account_status'].isin(['active', 'suspended'])].copy()
final = final.sort_values('first_mention_utc').reset_index(drop=True)

print(f'{before:,} accounts checked -> {len(final):,} kept ({before - len(final):,} dropped '
      'as deleted / forbidden / other)')
print()
print(final['account_status'].value_counts().to_string())
if SAMPLE_SIZE is not None:
    print(f'\nNOTE: SAMPLE_SIZE={SAMPLE_SIZE}, so this covers a sample of the '
          f'{len(suspicious):,} extracted accounts. Set SAMPLE_SIZE = None and re-run '
          'Step 5 to resolve them all.')
final.head(10)

## Step 6 — Tests on the final dataset

In [ ]:
# Schema is exactly what downstream attribute-collection will expect.
expected_columns = ['username', 'username_key', 'first_mention_utc', 'first_mention_date',
                    'times_mentioned', 'first_post_id', 'extraction_method', 'account_status']
assert list(final.columns) == expected_columns, f'unexpected columns: {list(final.columns)}'

# The brief: only active or suspended survive.
assert set(final['account_status']) <= {'active', 'suspended'}, 'only active/suspended may remain'

# One row per account, no nulls in any field.
assert final['username_key'].is_unique, 'one row per account'
assert not final.isna().any().any(), f'nulls present:\n{final.isna().sum()}'

# Counts and dates are sane.
assert final['times_mentioned'].min() >= 1, 'minimum mention count is 1'
assert (final['first_mention_date'] == pd.to_datetime(final['first_mention_utc'], unit='s')).all(), \
    'date column must agree with the raw timestamp'
corpus_start = pd.to_datetime(posts['created_utc'].min(), unit='s')
corpus_end = pd.to_datetime(posts['created_utc'].max(), unit='s')
assert final['first_mention_date'].between(corpus_start, corpus_end).all(), \
    'first mentions must fall inside the corpus date range'

# Every surviving account really does trace back to a post that names it.
for row in final.sample(min(25, len(final)), random_state=1).itertuples():
    src = posts.loc[posts['id'] == row.first_post_id].iloc[0]
    assert row.username_key in extract_suspects(src.title, src.selftext, VOCABULARY), \
        f'{row.username_key} not found in its cited first post'

print(f'all final-dataset tests passed on {len(final):,} accounts')

## Step 7 — Save

Parquet, because it is the right fit for "convenient to access, not computationally
intensive to read": columnar and compressed, so it loads several times faster and far
smaller than CSV; it round-trips dtypes, so `first_mention_date` comes back as a datetime
and `times_mentioned` as an integer with no re-parsing; and it supports column selection,
so the later attribute-collection step can read just `username_key` without touching the
rest. It is also what the rest of this project already uses.

The `data/` folder is git-ignored — it holds collected account data.

In [ ]:
final.to_parquet(OUTPUT_PARQUET, index=False, engine='pyarrow', compression='zstd')

size_kb = OUTPUT_PARQUET.stat().st_size / 1024
print(f'wrote {OUTPUT_PARQUET} ({size_kb:,.1f} KB, {len(final):,} rows)')

# Read it straight back and confirm the round-trip is lossless - a saved file that does
# not reload identically is worse than no file at all.
reloaded = pd.read_parquet(OUTPUT_PARQUET)
pd.testing.assert_frame_equal(final, reloaded)
print('round-trip verified: reloaded file is identical to the in-memory frame')

# Demonstrate the cheap partial read the next notebook will use.
keys_only = pd.read_parquet(OUTPUT_PARQUET, columns=['username_key', 'account_status'])
print(f'partial read of 2 columns: {len(keys_only):,} rows, '
      f'{keys_only.memory_usage(deep=True).sum()/1024:,.1f} KB in memory')
reloaded.head()

## Summary

`data/suspicious_usernames.parquet` holds one row per account accused in
r/TheseFuckingAccounts, with its first-mention date, how many separate posts named it, and
its current active/suspended status. `username_key` is the join key for the attribute
collection that comes next.

Re-run Step 5 with `SAMPLE_SIZE = None` to extend coverage to every extracted account; it
resumes from the checkpoint, so it can be run in as many sittings as needed.